In [1]:
import subprocess
subprocess.run([
    'pip', 'install', '-q',
    'datasets', 'lxml', 'cairosvg',
    'tokenizers', 'sentencepiece', 'tqdm', 'numpy'
], check=True)
print("Packages installed.")

Packages installed.


In [2]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [3]:
import os, re, json
from lxml import etree
from datasets import load_dataset
from tqdm import tqdm
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import ByteLevel
from tokenizers.processors import TemplateProcessing

BASE_DIR = '/content/drive/MyDrive/svg-lm-scaling'

DIRS = [
    'data/clean',
    'data/tokenized',
    'tokenizer',
    'checkpoints',
    'logs',
    'samples',
    'configs',
]

for d in DIRS:
    os.makedirs(f'{BASE_DIR}/{d}', exist_ok=True)

print(f"\nAll directories ready under: {BASE_DIR}")


All directories ready under: /content/drive/MyDrive/svg-lm-scaling


In [31]:
# Decoder-only Transformer model (nanoGPT-style) REDEFINITION
# Defines GPTConfig and GPT

import math
import torch
import torch.nn as nn
from torch.nn import functional as F
from dataclasses import dataclass

# Ref from https://github.com/karpathy/nanoGPT/blob/master/model.py#L29
# Taken directly from nanoGPT:
# - Overall model structure: GPT, Block, CausalSelfAttention, MLP class layout
# - Weight tying between wte (token embedding) and lm_head
# - The residual projection scaling: std = 0.02 / sqrt(2 * n_layer)
# - _init_weights normal(0, 0.02) for Linear, same for Embedding
# - AdamW with betas=(0.9, 0.95), weight_decay=0.1
# - Gradient clipping at max_norm=1.0
# - The flat .npy data format and get_batch random-offset sampling
# - Flash attention fallback pattern (using F.scaled_dot_product_attention if available, else manual masked attention)

# Modified from nanoGPT:
# - GPTConfig: added MODEL_CONFIGS dict for 5 fixed sizes instead of one (above)
# - num_params() added for the scaling plot
# - generate() added top-p (nucleus) sampling as nanoGPT only has top-k
# - Cosine LR schedule: nanoGPT has it inline in the training loop; we extracted it into a cosine_lr() function reused across steps
# - Training loop: added gradient accumulation, mixed precision (torch.amp), per-step metrics (tokens/s, GPU memory), and safe checkpointing
# ── Rebuild model architecture ────────────────────────────────

@dataclass
class GPTConfig:
    block_size: int = 512
    vocab_size: int = 4096
    n_layer:    int = 4
    n_head:     int = 4
    n_embd:     int = 128
    dropout:    float = 0.0
    bias:       bool = False

class CausalSelfAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        assert config.n_embd % config.n_head == 0
        self.c_attn  = nn.Linear(config.n_embd, 3 * config.n_embd, bias=config.bias)
        self.c_proj  = nn.Linear(config.n_embd, config.n_embd,     bias=config.bias)
        self.attn_drop  = nn.Dropout(config.dropout)
        self.resid_drop = nn.Dropout(config.dropout)
        self.n_head  = config.n_head
        self.n_embd  = config.n_embd
        self.dropout = config.dropout
        self.flash   = hasattr(F, 'scaled_dot_product_attention')
        if not self.flash:
            self.register_buffer('causal_mask',
                torch.tril(torch.ones(config.block_size, config.block_size))
                     .view(1, 1, config.block_size, config.block_size))
    def forward(self, x):
        B, T, C = x.size()
        hs = C // self.n_head
        q, k, v = self.c_attn(x).split(C, dim=2)
        q = q.view(B, T, self.n_head, hs).transpose(1, 2)
        k = k.view(B, T, self.n_head, hs).transpose(1, 2)
        v = v.view(B, T, self.n_head, hs).transpose(1, 2)
        if self.flash:
            y = F.scaled_dot_product_attention(q, k, v, dropout_p=self.dropout if self.training else 0.0, is_causal=True)
        else:
            att = (q @ k.transpose(-2, -1)) * (1.0 / math.sqrt(hs))
            att = att.masked_fill(self.causal_mask[:,:,:T,:T] == 0, float('-inf'))
            att = F.softmax(att, dim=-1)
            att = self.attn_drop(att)
            y = att @ v
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_drop(self.c_proj(y))

class MLP(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.c_fc   = nn.Linear(config.n_embd, 4 * config.n_embd, bias=config.bias)
        self.gelu   = nn.GELU()
        self.c_proj = nn.Linear(4 * config.n_embd, config.n_embd, bias=config.bias)
        self.drop   = nn.Dropout(config.dropout)
    def forward(self, x):
        return self.drop(self.c_proj(self.gelu(self.c_fc(x))))

class Block(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.ln_1 = nn.LayerNorm(config.n_embd, bias=config.bias)
        self.attn = CausalSelfAttention(config)
        self.ln_2 = nn.LayerNorm(config.n_embd, bias=config.bias)
        self.mlp  = MLP(config)
    def forward(self, x):
        x = x + self.attn(self.ln_1(x))
        x = x + self.mlp(self.ln_2(x))
        return x

class GPT(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.transformer = nn.ModuleDict(dict(
            wte  = nn.Embedding(config.vocab_size, config.n_embd),
            wpe  = nn.Embedding(config.block_size, config.n_embd),
            drop = nn.Dropout(config.dropout),
            h    = nn.ModuleList([Block(config) for _ in range(config.n_layer)]),
            ln_f = nn.LayerNorm(config.n_embd, bias=config.bias),
        ))
        self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)
        self.transformer.wte.weight = self.lm_head.weight
    def forward(self, idx, targets=None):
        B, T = idx.size()
        pos = torch.arange(T, device=idx.device)
        x = self.transformer.drop(self.transformer.wte(idx) + self.transformer.wpe(pos))
        for block in self.transformer.h:
            x = block(x)
        x = self.transformer.ln_f(x)
        if targets is not None:
            logits = self.lm_head(x)
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        else:
            logits = self.lm_head(x[:, [-1], :])
            loss = None
        return logits, loss
    def num_params(self):
        return sum(p.numel() for p in self.parameters()) - self.transformer.wte.weight.numel()
    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0, top_k=None, top_p=None,
                 eos_id=None, repetition_penalty=1.0, rep_window=64):
        n_prompt = idx.size(1)  # exclude prime tokens from penalty window
        for _ in range(max_new_tokens):
            idx_c = idx if idx.size(1) <= self.config.block_size else idx[:, -self.config.block_size:]
            logits, _ = self(idx_c)
            logits = logits[:, -1, :]
            if repetition_penalty != 1.0:
                # Only penalise tokens seen in the last rep_window *generated* tokens
                gen_start = max(n_prompt, idx.size(1) - rep_window)
                for token_id in set(idx[0, gen_start:].tolist()):
                    if logits[0, token_id] > 0:
                        logits[0, token_id] /= repetition_penalty
                    else:
                        logits[0, token_id] *= repetition_penalty
            logits = logits / temperature
            if top_k is not None:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = float('-inf')
            if top_p is not None:
                sl, si = torch.sort(logits, descending=True)
                cp = torch.cumsum(F.softmax(sl, dim=-1), dim=-1)
                sl[cp - F.softmax(sl, dim=-1) > top_p] = float('-inf')
                logits.scatter_(1, si, sl)
            nt = torch.multinomial(F.softmax(logits, dim=-1), num_samples=1)
            idx = torch.cat((idx, nt), dim=1)
            if eos_id is not None and nt.item() == eos_id:
                break
        return idx

In [32]:
# Sample generation + quantitative evaluation
# Saves generated SVGs to /samples/ and metrics to /results/eval_metrics.json

import os, json, math, time
import numpy as np
import torch
import torch.nn as nn
from torch.nn import functional as F
from dataclasses import dataclass

BASE_DIR = '/content/drive/MyDrive/svg-lm-scaling'
CKPT_DIR = f'{BASE_DIR}/checkpoints'
SAMPLE_DIR  = f'{BASE_DIR}/samples'
RESULTS_DIR = f'{BASE_DIR}/results'
TOK_DIR = f'{BASE_DIR}/tokenizer'
DATA_DIR = f'{BASE_DIR}/data/tokenized'
os.makedirs(SAMPLE_DIR, exist_ok=True)

# Config

# Choose the model with the lowest final_val_loss across all trained sizes/variants.
def pick_best_ckpt():
    candidates = []
    for result_file, prefix in [
        (f'{RESULTS_DIR}/mup_scaling_results.json', 'mup'),
        (f'{RESULTS_DIR}/scaling_results.json','sp'),
    ]:
        if not os.path.exists(result_file):
            continue
        with open(result_file) as f:
            results = json.load(f)
        for r in results:
            name = f"{prefix}_{r['name']}"
            path = f'{CKPT_DIR}/{name}.pt'
            if os.path.exists(path):
                candidates.append((r['final_val_loss'], name, path))

    if not candidates:
        raise FileNotFoundError("No checkpoints found. Run training")

    candidates.sort(key=lambda x: x[0])
    best_loss, best_name, best_path = candidates[0]
    print("Val losses across all checkpoints:")
    for loss, name, _ in candidates:
        marker = " best" if name == best_name else ""
        print(f"{name:<16}  val_loss={loss:.4f}{marker}")
    return best_path, best_name

CKPT_PATH, CKPT_NAME = pick_best_ckpt()
print(f"Using checkpoint: {CKPT_NAME}  ({CKPT_PATH})")

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {device}")


Val losses across all checkpoints:
sp_medium         val_loss=0.6322 best
sp_small          val_loss=0.6502
mup_large         val_loss=0.6557
mup_medium        val_loss=0.6774
mup_xl            val_loss=0.6944
mup_small         val_loss=0.6976
sp_tiny           val_loss=0.7226
mup_tiny          val_loss=0.7341
sp_large          val_loss=1.3587
sp_xl             val_loss=1.8051
Using checkpoint: sp_medium  (/content/drive/MyDrive/svg-lm-scaling/checkpoints/sp_medium.pt)
Device: cuda


In [33]:
# Load checkpoint

ckpt = torch.load(CKPT_PATH, map_location=device)
cfg_dict = ckpt['config']
config = GPTConfig(**cfg_dict)

# Handle mup checkpoint: MuReadout has different output layer
# rebuild as standard GPT (weights are compatible for generation)
model = GPT(config).to(device)
state = ckpt['model_state']
# Strip '_orig_mod.' prefix from compiled model keys
state = {k.replace('_orig_mod.', ''): v for k, v in state.items()}
# Handle mup lm_head, standard lm_head shape mismatch gracefully
try:
    model.load_state_dict(state, strict=True)
except RuntimeError:
    model.load_state_dict(state, strict=False)
    print("Note: partial weight load (mup checkpoint — some keys skipped)")
model.eval()
print(f"Model loaded: {model.num_params():,} params")

# Load tokenizer

from tokenizers import Tokenizer
from tokenizers.decoders import ByteLevel as ByteLevelDecoder
tokenizer = Tokenizer.from_file(f'{TOK_DIR}/svg_bpe.json')
tokenizer.decoder = ByteLevelDecoder()   # fixes G/C markers in decoded output
with open(f'{TOK_DIR}/token_config.json') as f:
    tok_cfg = json.load(f)

BOS_ID = tok_cfg['bos_id']
EOS_ID = tok_cfg['eos_id']

def encode(text):
    # add_special_tokens=False skips the TemplateProcessing post-processor
    # to not get double BOS/EOS when building prompt prefixes
    return tokenizer.encode(text, add_special_tokens=False).ids

def decode(ids):
    return tokenizer.decode(ids)


# Generation helpers

# End the prime inside a <path d="..."> so the model must generate coordinates
# rather than being free to predict arbitrary subword tokens after the header.
SVG_PRIME = '<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 24 24"><path d="M'

def generate_svg(prompt_text=None, max_new_tokens=400, temperature=1.0,
                 top_k=50, top_p=None, repetition_penalty=1.5):
    """
    Generate one SVG string. prompt_text=None for unconditional.
    """
    prime = prompt_text if prompt_text is not None else SVG_PRIME
    prefix_ids = [BOS_ID] + encode(prime)

    idx = torch.tensor([prefix_ids], dtype=torch.long, device=device)
    with torch.no_grad():
        out = model.generate(idx, max_new_tokens=max_new_tokens,
                             temperature=temperature, top_k=top_k,
                             top_p=top_p, eos_id=EOS_ID,
                             repetition_penalty=repetition_penalty)
    ids = out[0].tolist()
    # Trim BOS and EOS
    if BOS_ID in ids:
        ids = ids[ids.index(BOS_ID)+1:]
    if EOS_ID in ids:
        ids = ids[:ids.index(EOS_ID)]
    return decode(ids)


# Unconditional samples

print("\n" + "="*60)
print("Generating 10 unconditional SVG samples (temperature=0.8, top_k=50)")
print("="*60)

unconditional_samples = []
for i in range(10):
    svg = generate_svg(temperature=1.0, top_k=50)
    unconditional_samples.append(svg)
    print(f"\n Sample {i+1}")
    print(svg[:200] + ('…' if len(svg) > 200 else ''))

with open(f'{SAMPLE_DIR}/unconditional.json', 'w') as f:
    json.dump(unconditional_samples, f, indent=2)
print(f"\nSaved 10 unconditional samples to {SAMPLE_DIR}/unconditional.json")

Model loaded: 10,818,432 params

Generating 10 unconditional SVG samples (temperature=0.8, top_k=50)

 Sample 1
<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 24 24"><path d="M20.4 21 L16 22 L11.5 18.8"/></svg>

 Sample 2
<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 24 24"><path d="M6.5 24 L-4.8 23 L2.3 22"/> <path fill="none" stroke="black" stroke-width=".3" stroke-opacity="1.0" filling="0" d="M9 21.7 L14.3 17…

 Sample 3
<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 24 24"><path d="M1.2 5 L24.9 9 C25.6 10 25.7 11 24.6 12.5 L18 16 C16.4 17 15 18 13.3 19.2 L13 20.8 C12 21 23 22 7.9 21.6 8 14.7 14 17.3 L14.1 15.3 …

 Sample 4
<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 24 24"><path d="M8.1 24.3 L8 25.4 C9 26 10 27.6 11 28.6 L11 29.4 L12 22.7 C10 23 8.6 21.5 7 20.2 L6 19.1 C7 17 9 15.1 11 14.2 C13 13 15 12.7 16 14 …

 Sample 5
<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 24 24"><path d="M21.5 10 L18.6 12 C17 15 16 17 14 19 11 18.2 L9 22.8 C7 23.1 7.3

In [34]:
# Temperature comparison

print("\n" + "="*60)
print("Temperature comparison on same seed (top_k=50)")
print("="*60)

temp_samples = {}
for temp in [0.5, 0.8, 1.0]:
    torch.manual_seed(42)
    svg = generate_svg(temperature=temp, top_k=50)
    temp_samples[str(temp)] = svg
    print(f"\ntemperature={temp}")
    print(svg[:200] + ('…' if len(svg) > 200 else ''))

with open(f'{SAMPLE_DIR}/temperature_comparison.json', 'w') as f:
    json.dump(temp_samples, f, indent=2)


Temperature comparison on same seed (top_k=50)

temperature=0.5
<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 24 24"><path d="M8.1 24 L7.3 23 C6.9 22 6 21 5 19.2 C5 18.4 4 16.5 3 14.1 C2 12 2 9.9 1.2 8.1 L1 6"/> <path fill="none" stroke="black" stroke-width…

temperature=0.8
<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 24 24"><path d="M3.1 0.2 L3.9 1.6 C4.7 2 5 3 6 4.6 C5.8 7.1 5 9 5 10 4 11.2 C4 12.4 4 14.5 4 16.3 L10.8 13.8 L17 8.1 C18.1 6 19.7 4 21.9 1.8 L21.9 …

temperature=1.0
<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 24 24"><path d="M11.1 23 L11 18.4 C10 16 9 13 6.7 11.9 6.6 14.3 L5 17 L1.8 21 L2.7 20.1 C2 19 2 17.5 1.6 16.1 L0.9 12 L3 7.1 C2.1 5 3.7 4 5.4 3.8 L…


In [35]:
# Top-k vs nucleus sampling

print("\n" + "="*60)
print("Sampling strategy comparison")
print("="*60)

torch.manual_seed(0)
svg_topk = generate_svg(temperature=0.8, top_k=50,  top_p=None)
torch.manual_seed(0)
svg_nucleus = generate_svg(temperature=0.8, top_k=None, top_p=0.9)
torch.manual_seed(0)
svg_greedy = generate_svg(temperature=0.01, top_k=1)

print(f"top-k=50: {svg_topk[:150]}…")
print(f"nucleus p=0.9: {svg_nucleus[:150]}…")
print(f"near-greedy: {svg_greedy[:150]}…")

sampling_results = {'top_k_50': svg_topk, 'nucleus_p0.9': svg_nucleus, 'greedy': svg_greedy}
with open(f'{SAMPLE_DIR}/sampling_strategies.json', 'w') as f:
    json.dump(sampling_results, f, indent=2)

# Prefix-conditioned samples

print("\n" + "="*60)
print("Generating 5 prefix-conditioned samples")
print("="*60)

PREFIXES = [
    ('<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 24 24"><circle cx="12" cy="8" r="4"/>',
     "Partial face (circle for head) to check if model adds features"),
    ('<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 24 24"><path d="M2 12',
     "Open path to check if model closes it"),
    ('<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 24 24"><g fill="none" stroke="currentColor"><rect x="3" y="3" width="18" height="18" rx="2"/>',
     "Group with one rectangle to check if model adds related shapes"),
    ('<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 24 24"><path d="M12 2L2 7',
     "Arrow start to see if model completes the arrow"),
    ('<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 24 24" fill="currentColor"><path d="M12 17.27L18.18 21',
     "Star partial"),
]

prefix_results = []
for prefix, description in PREFIXES:
    svg = generate_svg(prompt_text=prefix, temperature=0.8, top_k=50)
    full_svg = prefix + svg   # prepend the original prefix
    prefix_results.append({'prefix': prefix, 'completion': svg,
                           'full_svg': full_svg, 'description': description})
    print(f"\n {description}")
    print(f"Prefix:     {prefix[:80]}…")
    print(f"Completion: {svg[:120]}…")

with open(f'{SAMPLE_DIR}/prefix_conditioned.json', 'w') as f:
    json.dump(prefix_results, f, indent=2)
print(f"\nSaved prefix-conditioned samples → {SAMPLE_DIR}/prefix_conditioned.json")


Sampling strategy comparison
top-k=50: <svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 24 24"><path d="M5.8 24.1 L6.9 22.7 C8 22 8 21.4 9 20.2 L10 14.3 C9 13 7 14 6.3 15.3 5 16.4 C4 17…
nucleus p=0.9: <svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 24 24"><path d="M5.8 24.1 L6.9 22.7 C8 21 8 19 7 17.2 C7 16 5 14 4 12.4 3 11.3 C2 10 1.3 9.4 2.5 …
near-greedy: <svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 24 24"><path d="M8.5 24 L-6.9 25 C7.2 26 6 27 5.3 28.1 C4 29 3 30 2.5 31.2 C0.9 32 4.2 33.2 7 34.…

Generating 5 prefix-conditioned samples

 Partial face (circle for head) to check if model adds features
Prefix:     <svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 24 24"><circle cx="12" cy="…
Completion: <svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 24 24"><circle cx="12" cy="8" r="4"/> <path fill="none" stroke="bla…

 Open path to check if model closes it
Prefix:     <svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 24 24"><path d="M2 12…
Completion: <s

In [36]:
# Quantitative evaluation

print("\n" + "="*60)
print("Quantitative Evaluation")
print("="*60)

# Perplexity on test set
test_data = np.load(f'{DATA_DIR}/test.npy', mmap_mode='r')
print("\nComputing test set perplexity")

def compute_perplexity(model, data, block_size, device, n_batches=200, batch_size=16):
    model.eval()
    total_loss = 0.0
    total_tokens = 0
    with torch.no_grad():
        for _ in range(n_batches):
            ix = torch.randint(len(data) - block_size, (batch_size,))
            x = torch.stack([torch.from_numpy(data[i:i+block_size].astype(np.int64)) for i in ix]).to(device)
            y = torch.stack([torch.from_numpy(data[i+1:i+1+block_size].astype(np.int64)) for i in ix]).to(device)
            _, loss = model(x, y)
            total_loss += loss.item() * x.numel()
            total_tokens += x.numel()
    avg_loss = total_loss / total_tokens
    return math.exp(avg_loss), avg_loss

perplexity, avg_nll = compute_perplexity(model, test_data, config.block_size, device)
print(f"  Test perplexity: {perplexity:.2f}  (avg NLL: {avg_nll:.4f})")

# Validity metrics on generated samples
try:
    from lxml import etree
    lxml_ok = True
except ImportError:
    print("lxml not installed; skipping XML validation (run: pip install lxml)")
    lxml_ok = False

try:
    import cairosvg
    cairo_ok = True
except ImportError:
    print("cairosvg not installed; skipping render check (run: pip install cairosvg)")
    cairo_ok = False

print("\nGenerating 50 samples for validity evaluation")
eval_samples = []
for _ in range(50):
    svg = generate_svg(temperature=0.8, top_k=50)
    eval_samples.append(svg)


Quantitative Evaluation

Computing test set perplexity
  Test perplexity: 1.87  (avg NLL: 0.6279)

Generating 50 samples for validity evaluation


In [37]:
def check_xml_valid(svg_str):
    if not lxml_ok:
        return None
    try:
        etree.fromstring(svg_str.encode('utf-8'))
        return True
    except Exception:
        return False

def check_svg_renders(svg_str):
    if not cairo_ok:
        return None
    try:
        cairosvg.svg2png(bytestring=svg_str.encode('utf-8'))
        return True
    except Exception:
        return False

def check_structural(svg_str):
    """
    Check <svg> root, closed tags, viewBox
    """
    s = svg_str.strip()
    has_svg_root = s.startswith('<svg') or s.startswith('<?xml')
    has_close = '</svg>' in s
    has_viewbox = 'viewBox' in s or 'viewbox' in s
    return has_svg_root and has_close, has_svg_root, has_close, has_viewbox

xml_valid = [check_xml_valid(s)    for s in eval_samples]
svg_rendered = [check_svg_renders(s)  for s in eval_samples]
structural = [check_structural(s)   for s in eval_samples]

xml_rate = sum(v for v in xml_valid    if v is not None) / max(sum(v is not None for v in xml_valid), 1)
render_rate = sum(v for v in svg_rendered if v is not None) / max(sum(v is not None for v in svg_rendered), 1)
struct_rate = sum(v[0] for v in structural) / len(structural)
viewbox_rate = sum(v[3] for v in structural) / len(structural)

print(f"\nValidity metrics (over {len(eval_samples)} generated samples):")
print(f"XML validity rate: {xml_rate*100:.1f}%"
      + (" (lxml)" if lxml_ok else " (lxml not available)"))
print(f" SVG render rate: {render_rate*100:.1f}%"
      + (" (cairosvg)" if cairo_ok else " (cairosvg not available)"))
print(f"Structural validity: {struct_rate*100:.1f}%  (<svg> root + </svg> close)")
print(f"viewBox usage: {viewbox_rate*100:.1f}%")

avg_len = np.mean([len(s) for s in eval_samples])
print(f"Average generated length: {avg_len:.0f} chars")


Validity metrics (over 50 generated samples):
XML validity rate: 98.0% (lxml)
 SVG render rate: 98.0% (cairosvg)
Structural validity: 98.0%  (<svg> root + </svg> close)
viewBox usage: 100.0%
Average generated length: 320 chars


In [38]:
# Save all metrics
metrics = {
    'checkpoint': CKPT_NAME,
    'perplexity': perplexity,
    'avg_nll': avg_nll,
    'n_eval_samples': len(eval_samples),
    'xml_validity_rate': xml_rate if lxml_ok else None,
    'svg_render_rate':   render_rate if cairo_ok else None,
    'structural_validity_rate': struct_rate,
    'viewbox_rate': viewbox_rate,
    'avg_generated_length': avg_len,
}
with open(f'{RESULTS_DIR}/eval_metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)
print(f"\nSaved metrics to {RESULTS_DIR}/eval_metrics.json")

# Render generated samples as HTML

import re as _re

def sanitize_svg(svg, w=120, h=120):
    """Best-effort cleanup so inline SVG has a chance to render."""
    s = svg.strip()
    # Model sometimes emits a stray leading > right after the prime's closing >
    s = s.lstrip('> \n')
    # Ensure it starts with an <svg opening tag
    if not s.startswith('<svg'):
        s = f'<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 24 24">{s}'
    # Inject width/height so the browser knows the intrinsic size
    if 'width=' not in s[:120]:
        s = _re.sub(r'<svg\b', f'<svg width="{w}" height="{h}"', s, count=1)
    # Ensure the SVG is closed
    if not s.rstrip().endswith('</svg>'):
        # Truncate at the last complete tag to avoid unclosed attributes
        last_close = s.rfind('>')
        if last_close != -1:
            s = s[:last_close + 1]
        s = s + '</svg>'
    return s

# Inline SVG goes straight into the HTML

html_rows = []
all_display = unconditional_samples[:10] + [r['full_svg'] for r in prefix_results]
labels = [f"Unconditional {i+1}" for i in range(10)] + [r['description'] for r in prefix_results]

for label, svg in zip(labels, all_display):
    clean = sanitize_svg(svg)
    html_rows.append(
        f'<div style="display:inline-block;margin:8px;text-align:center;vertical-align:top">'
        f'<div style="font-size:11px;width:120px;margin-bottom:4px">{label}</div>'
        f'<div style="width:120px;height:120px;border:1px solid #ccc;overflow:hidden">{clean}</div>'
        f'</div>'
    )

html_out = '<html><body style="font-family:monospace;padding:16px">' + ''.join(html_rows) + '</body></html>'
html_path = f'{SAMPLE_DIR}/samples_grid.html'
with open(html_path, 'w') as f:
    f.write(html_out)
print(f"Saved HTML grid to {html_path}")

# Show inline in Colab
try:
    from IPython.display import HTML, display
    display(HTML(html_out))
except Exception:
    pass

print("\nAll results saved to:", RESULTS_DIR)
print("Final deliverables:")
print(f"Checkpoints: {CKPT_DIR}/sp_*.pt  and  mup_*.pt")
print(f"Results: {RESULTS_DIR}/")
print(f"Samples: {SAMPLE_DIR}/")


Saved metrics to /content/drive/MyDrive/svg-lm-scaling/results/eval_metrics.json
Saved HTML grid to /content/drive/MyDrive/svg-lm-scaling/samples/samples_grid.html



All results saved to: /content/drive/MyDrive/svg-lm-scaling/results
Final deliverables:
Checkpoints: /content/drive/MyDrive/svg-lm-scaling/checkpoints/sp_*.pt  and  mup_*.pt
Results: /content/drive/MyDrive/svg-lm-scaling/results/
Samples: /content/drive/MyDrive/svg-lm-scaling/samples/
